In [67]:
import pandas as pd
from pathlib import Path
import duckdb
import sys
from openpyxl.writer.excel import ExcelWriter

In [74]:
folder = Path("samples/shopee")
vba_df = Path(folder / "vba/SHOPEE CODE.xlsm")
order_df = pd.read_excel(folder /'đơn tải sàn.xlsx')

In [75]:
combo_df_rn = {
    'Tên sản phẩm': 'combo_name',
    'Tên phân loại hàng': 'combo_variant',
    'Giá ưu đãi': 'total_price',
    'Số loại sp': 'combo_product_count',
    'Tên sản phẩm khi tách': 'product_name',
    'Giá ưu đãi khi tách': 'product_price',
    'số lượng khi tách': 'product_quantity',
}

code_df_rn = {
    'Mã CODE': 'product_code',
    'Tên SP': 'product_name'
}

def read_sheet(path, sn: str, rn_dict: dict):
    old_cols = [col for col in rn_dict.keys()]
    new_cols = [col for col in rn_dict.values()]
    df = pd.read_excel(path, sheet_name=sn, usecols=old_cols)
    df_rename = df.rename(columns=rn_dict)
    df_final = df_rename[new_cols]
    return df_final

combo_df = read_sheet(vba_df, "DLCOMBO", combo_df_rn)
code_df = read_sheet(vba_df, "CODE", code_df_rn)

In [ ]:
product = duckdb.sql("""
    SELECT DISTINCT
    "product_code",
    "product_name"
    FROM code_df
""")

dlcombo_extract = duckdb.sql("""
    SELECT
        cb.combo_name,
        NULLIF(cb.combo_variant, '0') as combo_variant,
        pl.product_code,
        pl.product_name,
        cb.combo_product_count,
        cb.product_price,
        cb.product_quantity,
        cb.product_price * cb.product_quantity as total_byrow,
        cb.total_price as total_value,

        SUM(cb.product_price * cb.product_quantity)
            OVER (
            PARTITION BY cb.combo_name, cb.combo_variant, cb.total_price
            ) as total_recalculated,

        COUNT(cb.combo_name)
            OVER(
            PARTITION BY cb.combo_name, cb.combo_variant, cb.total_price
            ) AS product_count
    FROM combo_df cb
    JOIN product pl USING (product_name)
    -- Lọc bỏ các product_code không khớp với combo và các mã sản phẩm = 0
    WHERE
        pl.product_code IS NOT NULL
        AND pl.product_code <> 0
    ORDER BY combo_name, total_value
""").to_df()

# Trích xuất các combo có trong VBA
combo = duckdb.sql("""
    SELECT
        row_number() OVER() - 1 as combo_key,
        combo_name,
        combo_variant,
        total_recalculated
    FROM (
        SELECT DISTINCT combo_name, combo_variant, total_recalculated
        FROM dlcombo_extract
        ORDER BY 1, 2, 3
    )
""").to_df()

# Trích xuất danh sách sản phẩm
products = duckdb.sql("""
    SELECT DISTINCT
    product_code,
    product_name
    FROM dlcombo_extract
    ORDER BY product_code
""").to_df()

# Bảng chi tiết: Ánh xạ sản phẩm vào từng combo_key
combo_details = duckdb.sql("""
    SELECT
        c.combo_key,
        cd.product_code,
        cd.product_name,
        cd.product_price,
        cd.product_quantity
    FROM dlcombo_extract cd
    LEFT JOIN combo c
        ON cd.combo_name IS NOT DISTINCT FROM c.combo_name
        AND cd.combo_variant IS NOT DISTINCT FROM c.combo_variant
        AND cd.total_recalculated = c.total_recalculated
    ORDER BY c.combo_key, cd.product_code
""").to_df()

In [ ]:
# Các combo bị điền sai thông tin về tổng số lượng sản phẩm
# Các dòng này sẽ bị drop đi vì cài đặt sai
filter = dlcombo_extract['combo_product_count'] != dlcombo_extract['product_count']
false_product = dlcombo_extract[filter].loc[:, ['combo_name', 'combo_variant', 'total_value', 'product_code', 'product_price', 'combo_product_count', 'product_count']]

In [ ]:
# Tạo bảng combo - product code hoàn chỉnh
new_master_data = duckdb.sql("""
    SELECT
    combo_name,
    combo_variant,
    product_code,
    product_name,
    product_quantity,
    product_price,
    total_recalculated
    FROM combo_details
    JOIN combo USING (combo_key)
""").to_df()

<h3> Tải file control

In [ ]:
order_df = order_df.rename(columns=rename_dict)
join_df = duckdb.sql("""
    WITH order_processed AS (
        SELECT
            order_id,
            product_name AS combo_name,
            variation_name AS combo_variant,
            quantity,
            deal_price,
            SUM(deal_price * quantity) OVER (PARTITION BY od.order_id) as total_recalculated
        FROM order_df od
    )
    SELECT *, ms.total_recalculated FROM order_processed op
    LEFT JOIN new_master_data ms USING (combo_name, combo_variant, total_recalculated)
    WHERE product_code IS NULL
""").to_df()

NameError: name 'rename_dict' is not defined

In [ ]:
mapping_df = pd.read_excel('shopee_control_file.xlsx', sheet_name='column_mapping')

In [ ]:
rename_dict = dict(zip(mapping_df['raw_name'], mapping_df['sys_name']))

<h3> Tải file hoá đơn

In [86]:
import pandas as pd

# 1. Tạo dữ liệu gốc
data = {
    'Khu vực': ['Miền Bắc', 'Miền Bắc', 'Miền Bắc', 'Miền Nam', 'Miền Nam', 'Miền Nam'],
    'Ngày': ['01/05', '01/05', '02/05', '01/05', '01/05', '02/05'],
    'Sản phẩm': ['Laptop', 'iPhone', 'Laptop', 'Laptop', 'iPhone', 'iPhone'],
    'Doanh thu': [100, 200, 150, 120, 300, 250]
}
df = pd.DataFrame(data)

# 2. Tạo Pivot Table cơ bản (chưa có Subtotal)
pivot = df.pivot_table(
    index=['Khu vực', 'Ngày'], 
    columns='Sản phẩm', 
    values='Doanh thu', 
    aggfunc='sum', 
    fill_value=0
)

# 3. Tính Subtotal cho mỗi Khu vực
# Chúng ta nhóm theo cấp index đầu tiên (Khu vực) và tính tổng
subtotals = pivot.groupby(level=0).sum()

# Đặt tên cho index của dòng Subtotal (ví dụ: 'Miền Bắc Total')
subtotals.index = pd.MultiIndex.from_tuples([(x, 'Subtotal') for x in subtotals.index])

# 4. Gộp bảng Pivot và bảng Subtotal lại, sau đó sắp xếp
final_table = pd.concat([pivot, subtotals]).sort_index()

# 5. Thêm Grand Total (Tổng cuối cùng)
grand_total = final_table.sum()
final_table.loc[('Grand Total', ''), :] = grand_total

print(final_table)

# Xuất ra Excel
final_table.to_excel('pivot_with_subtotals.xlsx')

Sản phẩm              Laptop  iPhone
Miền Bắc    01/05      100.0   200.0
            02/05      150.0     0.0
            Subtotal   250.0   200.0
Miền Nam    01/05      120.0   300.0
            02/05        0.0   250.0
            Subtotal   120.0   550.0
Grand Total            740.0  1500.0
